In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.getcwd())

import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Pagella", "Book Antiqua", "Palatino Linotype", "DejaVu Serif"]
})

import pickle
import jax
import jax.numpy as jnp

from conditional_diffusion_3d_model import ConditionalUNet3D, DiffusionModelConfig
from train_conditional_diffusion import preload_hdf5_to_memory, train_model

### Import validation data

In [ ]:
val_path = "/projects/mccleary_group/habjan.e/TNG/Data/conditional_diffusion_data/cond_diffusion_16cubed_16img_test.h5"
data_dict = preload_hdf5_to_memory(val_path)

### Import samples

In [ ]:
suffix = "_16cube_16img_v10"

sample_base_path = '/projects/mccleary_group/habjan.e/TNG/Data/conditional_diffusion_data/cd_examples/'
sample_path = sample_base_path + f'sampled_cubes_50random_fixedseed_0{suffix}.npz'
data = np.load(sample_path)
data_im, data_true_cube, data_sampled_cubes, seed = data['conditioning_images'], data['true_cubes'], data['sampled_cubes'], data['selection_seed']

### Plot data

In [ ]:
idx = 2

imgs = data_im[idx]
cmap = 'cubehelix'

fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(15, 10), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.2, wspace=0.1, hspace=0.1)
ax1, ax2, ax3, ax4 = axes[0], axes[1], axes[2], axes[3]
#ax1, ax2, ax3 = axes[0], axes[1], axes[2]

im1 = ax1.imshow(imgs[0], vmin = np.quantile(imgs[0], 0.5), vmax = np.quantile(imgs[0], 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

im2 = ax2.imshow(imgs[1], cmap=cmap, origin="upper")
ax2.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax2.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.02)
cbar.set_label("Galaxy Density", fontsize=14, fontweight='semibold')

im3 = ax3.imshow(imgs[2], cmap=cmap, origin="upper")
ax3.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax3.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.02)
cbar.set_label("$\mathbf{V_{z}}$", fontsize=14, fontweight='semibold')

im4 = ax4.imshow(imgs[3], cmap=cmap, origin="upper")
ax4.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax4.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im4, ax=ax4, fraction=0.046, pad=0.02)
cbar.set_label("$\mathbf{\sigma_{v_z}}$", fontsize=14, fontweight='semibold')

fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/conddiff_data.png", bbox_inches="tight")
plt.show()

### XY-projected data cube (this is the same as the conditional projected mass)

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

log10_rho = data_true_cube[idx] * cube_std + cube_mean
rho = 10**log10_rho

rho_im = np.sum(rho, axis = 0)

imgs = (np.log10(rho_im ) - data_dict["metadata"]["proj_mass_log10_mean"]) / data_dict["metadata"]["proj_mass_log10_std"]
cmap = 'cubehelix'

fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(5, 5), constrained_layout=False)
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.2, wspace=-0.1, hspace=0.1)

im1 = ax1.imshow(imgs, vmin = np.quantile(imgs, 0.5), vmax = np.quantile(imgs, 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

### Projected true density cube in zx-plane

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

log10_rho = data_true_cube[idx] * cube_std + cube_mean
rho = 10**log10_rho

rho_im = np.sum(rho, axis = 1)

imgs = (np.log10(rho_im ) - data_dict["metadata"]["proj_mass_log10_mean"]) / data_dict["metadata"]["proj_mass_log10_std"]
cmap = 'cubehelix'

fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(5, 5), constrained_layout=False)
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.2, wspace=-0.1, hspace=0.1)

im1 = ax1.imshow(imgs, vmin = np.quantile(imgs, 0.5), vmax = np.quantile(imgs, 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("Z-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

### XY-plane of sampled cube

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

log10_rho = data_sampled_cubes[idx] * cube_std + cube_mean
rho = 10**log10_rho

rho_im = np.sum(rho, axis = 0)

imgs = (np.log10(rho_im ) - data_dict["metadata"]["proj_mass_log10_mean"]) / data_dict["metadata"]["proj_mass_log10_std"]
cmap = 'cubehelix'

fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(5, 5), constrained_layout=False)
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.2, wspace=-0.1, hspace=0.1)

im1 = ax1.imshow(imgs, vmin = np.quantile(imgs, 0.5), vmax = np.quantile(imgs, 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("Y-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

### ZX-plane of sampled cube

In [ ]:
cube_mean = data_dict["metadata"]["cube_log10_mean"]
cube_std  = data_dict["metadata"]["cube_log10_std"]

log10_rho = data_sampled_cubes[idx] * cube_std + cube_mean
rho = 10**log10_rho

rho_im = np.sum(rho, axis = 1)

imgs = (np.log10(rho_im ) - data_dict["metadata"]["proj_mass_log10_mean"]) / data_dict["metadata"]["proj_mass_log10_std"]
cmap = 'cubehelix'

fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(5, 5), constrained_layout=False)
fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.2, wspace=-0.1, hspace=0.1)

im1 = ax1.imshow(imgs, vmin = np.quantile(imgs, 0.5), vmax = np.quantile(imgs, 0.98), cmap=cmap, origin="upper")
ax1.set_xlabel("X-coordinate", fontsize=14, fontweight='semibold')
ax1.set_ylabel("Z-coordinate", fontsize=14, fontweight='semibold')
cbar = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.02)
cbar.set_label("Projected Mass", fontsize=14, fontweight='semibold')

### Function for calculating cluster mass from voxelized cube

In [ ]:
def cube_to_mass_msun(cube_norm, metadata):
    """
    cube_norm: (Z,Y,X) normalized/log-standardized cube from your dataset or samples
    metadata: HDF5 attrs dict from preload_hdf5_to_memory(... )["metadata"]

    returns:
        enclosed mass inside the cube in Msun
    """
    cube_mean = float(metadata["cube_log10_mean"])
    cube_std = float(metadata["cube_log10_std"])
    floor_value = float(metadata["floor_value"])
    fov_mpc = float(metadata["map_fov_mpc"])
    N = int(metadata["cube_resolution"])

    # invert normalization
    rho = np.zeros_like(cube_norm, dtype=np.float64)
    mask = cube_norm > floor_value + 1e-6
    rho[mask] = 10.0 ** (cube_norm[mask] * cube_std + cube_mean)  # Msun / Mpc^3

    # voxel volume
    voxel_size = (2.0 * fov_mpc) / N
    voxel_vol = voxel_size ** 3  # Mpc^3

    mass = np.sum(rho) * voxel_vol
    return mass

### Make arrays of cluster enclosed mass

In [ ]:
metadata = data_dict["metadata"]

true_masses = np.log10(np.array([cube_to_mass_msun(data_true_cube[i, :, :, :], metadata) for i in range(data_true_cube.shape[0])]))
sample_masses = np.log10(np.array([cube_to_mass_msun(data_sampled_cubes[i, :, :, :], metadata) for i in range(data_sampled_cubes.shape[0])]))

### Plot direct comparison of cube masses

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(true_masses, sample_masses, c='deepskyblue')

concat_mass = np.concatenate([true_masses, sample_masses])
min_m, max_m = np.min(concat_mass)*0.99, np.max(concat_mass)*1.01

ax.set_xlim(min_m, max_m)
ax.set_ylim(min_m, max_m)


one_one = np.linspace(min_m * 0.1, max_m * 10, 25)
ax.plot(one_one, one_one, c='k')

ax.set_xlabel(r'True Cube Mass $[\log_{10}(M_{\rm cube}/M_\odot)]$', fontsize=15)
ax.set_ylabel(r'Sampled Cube $[\log_{10}(M_{\rm cube}/M_\odot)]$', fontsize=15)

fig.savefig("/home/habjan.e/TNG/cluster_deprojection/figures/cube_masses.png", bbox_inches="tight")

### Shapes function

In [ ]:
def cube_to_axis_lengths_mpc(cube_norm, metadata):
    """
    cube_norm: (Z,Y,X) normalized/log-standardized cube from your dataset or samples
    metadata: HDF5 attrs dict from preload_hdf5_to_memory(... )["metadata"]

    returns:
        a, b, c : characteristic mass-weighted axis lengths in Mpc,
                  ordered from largest to smallest
    """
    cube_mean = float(metadata["cube_log10_mean"])
    cube_std = float(metadata["cube_log10_std"])
    floor_value = float(metadata["floor_value"])
    fov_mpc = float(metadata["map_fov_mpc"])
    N = int(metadata["cube_resolution"])

    # invert normalization: rho in Msun / Mpc^3
    rho = np.zeros_like(cube_norm, dtype=np.float64)
    mask = cube_norm > floor_value + 1e-6
    rho[mask] = 10.0 ** (cube_norm[mask] * cube_std + cube_mean)

    # voxel geometry
    voxel_size = (2.0 * fov_mpc) / N
    voxel_vol = voxel_size ** 3

    # voxel masses in Msun
    mass = rho * voxel_vol
    total_mass = np.sum(mass)

    if total_mass <= 0:
        return np.nan, np.nan, np.nan

    # voxel-center coordinates in Mpc
    coords_1d = (np.arange(N, dtype=np.float64) + 0.5) * voxel_size - fov_mpc
    z, y, x = np.meshgrid(coords_1d, coords_1d, coords_1d, indexing="ij")

    # center of mass
    x_com = np.sum(mass * x) / total_mass
    y_com = np.sum(mass * y) / total_mass
    z_com = np.sum(mass * z) / total_mass

    dx = x - x_com
    dy = y - y_com
    dz = z - z_com

    # mass-weighted shape tensor
    S_xx = np.sum(mass * dx * dx) / total_mass
    S_yy = np.sum(mass * dy * dy) / total_mass
    S_zz = np.sum(mass * dz * dz) / total_mass
    S_xy = np.sum(mass * dx * dy) / total_mass
    S_xz = np.sum(mass * dx * dz) / total_mass
    S_yz = np.sum(mass * dy * dz) / total_mass

    S = np.array([
        [S_xx, S_xy, S_xz],
        [S_xy, S_yy, S_yz],
        [S_xz, S_yz, S_zz]
    ], dtype=np.float64)

    # diagonalize and sort largest -> smallest
    evals = np.linalg.eigvalsh(S)
    evals = np.sort(evals)[::-1]

    # characteristic axis lengths
    a, b, c = np.sqrt(np.clip(evals, 0.0, None))

    return float(a), float(b), float(c)

### Run shape code

In [ ]:
true_shapes = np.array([cube_to_axis_lengths_mpc(data_true_cube[i, :, :, :], metadata) for i in range(data_true_cube.shape[0])])
sample_shapes = np.array([cube_to_axis_lengths_mpc(data_sampled_cubes[i, :, :, :], metadata) for i in range(data_sampled_cubes.shape[0])])

### Plot shapes

In [ ]:
cmap = 'viridis'

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(13.5, 4), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.2, wspace=0.1, hspace=0.1)
ax1, ax2, ax3 = axes[0], axes[1], axes[2]

one_one = np.linspace(0, 10)

im1 = ax1.scatter(true_shapes[:, 0], sample_shapes[:, 0], c=true_masses, cmap=cmap)
ax1.plot(one_one, one_one, c='k')
ax1.set_xlabel(r"True $a$", fontsize=14, fontweight='semibold')
ax1.set_ylabel(r"Sampled $a$", fontsize=14, fontweight='semibold')

concat_shape = np.concatenate([true_shapes[:, 0], sample_shapes[:, 0]])
min_s, max_s = np.min(concat_shape)*0.95, np.max(concat_shape)*1.05
ax1.set_xlim(min_s, max_s)
ax1.set_ylim(min_s, max_s)

im2 = ax2.scatter(true_shapes[:, 1], sample_shapes[:, 1], c=true_masses, cmap=cmap)
ax2.plot(one_one, one_one, c='k')
ax2.set_xlabel(r"True $b$", fontsize=14, fontweight='semibold')
ax2.set_ylabel(r"Sampled $b$", fontsize=14, fontweight='semibold')

concat_shape = np.concatenate([true_shapes[:, 1], sample_shapes[:, 1]])
min_s, max_s = np.min(concat_shape)*0.95, np.max(concat_shape)*1.05
ax2.set_xlim(min_s, max_s)
ax2.set_ylim(min_s, max_s)

im3 = ax3.scatter(true_shapes[:, 2], sample_shapes[:, 2], c=true_masses, cmap=cmap)
ax3.plot(one_one, one_one, c='k')
ax3.set_xlabel(r"True $c$", fontsize=14, fontweight='semibold')
ax3.set_ylabel(r"Sampled $c$", fontsize=14, fontweight='semibold')

concat_shape = np.concatenate([true_shapes[:, 2], sample_shapes[:, 2]])
min_s, max_s = np.min(concat_shape)*0.95, np.max(concat_shape)*1.05
ax3.set_xlim(min_s, max_s)
ax3.set_ylim(min_s, max_s)

cbar = fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
cbar.set_label(r'Cube Mass $[\log_{10}(M_{\rm cube}/M_\odot)]$', fontsize=14, fontweight='semibold')

plt.show()